In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

CONFIGURACION

In [0]:

CATALOG = "smartclaims" 
BRONZE_SCHEMA = "bronce"
VOLUME_NAME = "landing"

spark = SparkSession.builder.getOrCreate()

BASE_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

BASE_PATH = "/FileStore/insurance"

# Rutas
CUSTOMERS_PATH = f"{BASE_PATH}/customers.csv"
POLICIES_PATH = f"{BASE_PATH}/policies.csv"
CLAIMS_PATH = f"{BASE_PATH}/claims.csv"
TELEMATICS_PATH = f"{BASE_PATH}/telematics"
TRAINING_IMAGES_PATH = f"{BASE_PATH}/training_images"
CLAIM_IMAGES_PATH = f"{BASE_PATH}/claim_images"

Asegurar schema Bronze


In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_SCHEMA}")

1. Leer archivos


In [0]:
customers_df = spark.read.option("header", True).option("inferSchema", True).csv(CUSTOMERS_PATH)
policies_df = spark.read.option("header", True).option("inferSchema", True).csv(POLICIES_PATH)
claims_df = spark.read.option("header", True).option("inferSchema", True).csv(CLAIMS_PATH)

telematics_df = spark.read.option("header", True).option("inferSchema", True).csv(TELEMATICS_PATH)

training_images_df = spark.read.format("binaryFile").load(TRAINING_IMAGES_PATH)
claim_images_df = spark.read.format("binaryFile").load(CLAIM_IMAGES_PATH)


2.Normalizar tipo de datos + ingest_timestamp

In [0]:

customers_bronze = customers_df.withColumn("ingestion_time", F.current_timestamp())
policies_bronze = policies_df.withColumn("ingestion_time", F.current_timestamp())
claims_bronze = claims_df.withColumn("ingestion_time", F.current_timestamp())
telematics_bronze = telematics_df.withColumn("ingestion_time", F.current_timestamp())
training_images_bronze = training_images_df.withColumn("ingestion_time", F.current_timestamp())
claim_images_bronze = claim_images_df.withColumn("ingestion_time", F.current_timestamp())

3. Metadata de imágenes

In [0]:
claim_images_metadata = (
    claim_images_df
    .select("path", "modificationTime", "length")
    .withColumn("image_name", F.element_at(F.split(F.col("path"), "/"), -1))
    .withColumn("image_id", F.monotonically_increasing_id())
    .withColumn("claim_no", F.lit(None))
    .withColumn("chassis_no", F.lit(None))
    .withColumn("ingestion_time", F.current_timestamp())
)

4. Escribir tablas Delta

In [0]:
customers_bronze.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

policies_bronze.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.policies")

claims_bronze.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claims")

telematics_bronze.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")

training_images_bronze.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.training_images")

claim_images_bronze.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")

claim_images_metadata.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata")

In [0]:
print("Tablas Bronze creadas correctamente:")
print(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
print(f"{CATALOG}.{BRONZE_SCHEMA}.policies")
print(f"{CATALOG}.{BRONZE_SCHEMA}.claims")
print(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")
print(f"{CATALOG}.{BRONZE_SCHEMA}.training_images")
print(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")
print(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata")